In [1]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import json
import time
import xarray as xr
import scipy.signal as signal
import glob
import os
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score, StratifiedKFold, permutation_test_score, GridSearchCV
from sklearn.metrics import classification_report, recall_score, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score,make_scorer,balanced_accuracy_score, f1_score

import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import seaborn as sns

from matplotlib.ticker import MultipleLocator


data_dir = '/home/lukeew/Local_work/data/01_SLW_xgb'
reference_dir = data_dir + '/CHCH_v3_reference/'
reference_mask_dir = reference_dir + 'processed{}/'
model_save_dir = reference_dir + 'models/'

model_mask_dir = data_dir + '/CHCH_v3_model_processed/{}/model_processed_{}/'


### Single day

In [2]:
date = '20210518'
model_name = 'model_001'

infile_ref = reference_mask_dir.format(date[:-2])+str(date)+'.nc'
print('  <--- ' + infile_ref)
ds_ref = xr.open_dataset(infile_ref)

infile_mod = model_mask_dir.format(model_name,date[:-2])+str(date)+'.nc'
print('  <--- ' + infile_mod)
ds_mod = xr.open_dataset(infile_mod)

  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_reference/processed202105/20210518.nc
  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_model_processed/model_001/model_processed_202105/20210518.nc


In [3]:
slwc_ref = ds_ref.slwc
slwc_t_ref = slwc_ref.max(dim='level')

slwc_t_mod = ds_mod.slwc_t
class_pks = ds_mod.classified_peaks
pks_t = ~class_pks.max(dim='level').isnull()  # True if peaks are present, false if no peaks present

In [32]:
print('{:.0f} cloudy time steps (with peaks detected), out of {} time steps ({:.1f}%)'.format(pks_t.sum(),len(pks_t),100*pks_t.mean()))
print('Ref: {:.0f} time steps have SLWC, out of {} total time steps ({:.1f}%)'.format(slwc_t_ref.sum(),len(slwc_t_ref),100*slwc_t_ref.mean()))
print('Ref: {:.0f} time steps have SLCC, out of {} cloudy time steps ({:.1f}%)'.format(slwc_t_ref[pks_t].sum(),len(slwc_t_ref[pks_t]),100*slwc_t_ref[pks_t].mean()))
print('Model: {:.0f} time steps have SLCC, out of {} total time steps ({:.1f}%)'.format(slwc_t_mod.sum(),len(slwc_t_mod),100*slwc_t_mod.mean()))
print('Model: {:.0f} time steps have SLCC, out of {} cloudy time steps ({:.1f}%)'.format(slwc_t_mod[pks_t].sum(),len(slwc_t_mod[pks_t]),100*slwc_t_mod[pks_t].mean()))

print('Precision/Recall of SLWC classification, examining all time steps: {:.3f}/{:.3f}'.format(precision_score(slwc_t_ref, slwc_t_mod),recall_score(slwc_t_ref, slwc_t_mod)))
print('Precision/Recall of SLWC classification, examining cloudy time steps: {:.3f}/{:.3f}'.format(precision_score(slwc_t_ref[pks_t], slwc_t_mod[pks_t]),recall_score(slwc_t_ref[pks_t], slwc_t_mod[pks_t])))


188 cloudy time steps (with peaks detected), out of 288 time steps (65.3%)
Ref: 148 time steps have SLWC, out of 288 total time steps (51.4%)
Ref: 145 time steps have SLCC, out of 188 cloudy time steps (77.1%)
Model: 145 time steps have SLCC, out of 288 total time steps (50.3%)
Model: 145 time steps have SLCC, out of 188 cloudy time steps (77.1%)
Precision/Recall of SLWC classification, examining all time steps: 0.910/0.892
Precision/Recall of SLWC classification, examining cloudy time steps: 0.910/0.910


In [23]:
print(classification_report(slwc_t_ref, slwc_t_mod))
print(precision_score(slwc_t_ref, slwc_t_mod))
print(recall_score(slwc_t_ref, slwc_t_mod))
print(confusion_matrix(slwc_t_ref, slwc_t_mod))


              precision    recall  f1-score   support

           0       0.89      0.91      0.90       140
           1       0.91      0.89      0.90       148

    accuracy                           0.90       288
   macro avg       0.90      0.90      0.90       288
weighted avg       0.90      0.90      0.90       288

0.9103448275862069
0.8918918918918919
[[127  13]
 [ 16 132]]


In [28]:
print(classification_report(slwc_t_ref[pks_t], slwc_t_mod[pks_t]))
print(precision_score(slwc_t_ref[pks_t], slwc_t_mod[pks_t]))
print(recall_score(slwc_t_ref[pks_t], slwc_t_mod[pks_t]))
print(confusion_matrix(slwc_t_ref[pks_t], slwc_t_mod[pks_t]))


              precision    recall  f1-score   support

           0       0.70      0.70      0.70        43
           1       0.91      0.91      0.91       145

    accuracy                           0.86       188
   macro avg       0.80      0.80      0.80       188
weighted avg       0.86      0.86      0.86       188

0.9103448275862069
0.9103448275862069
[[ 30  13]
 [ 13 132]]


### Multiple Days

In [57]:
def printStats(slwc_t_ref, slwc_t_mod, pks_t):
    
    ## Computing stats
    print('{:.0f} cloudy time steps (with peaks detected), out of {} time steps ({:.1f}%)'.format(pks_t.sum(),len(pks_t),100*pks_t.mean()))
    print('Ref: {:.0f} time steps have SLWC, out of {} total time steps ({:.1f}%)'.format(slwc_t_ref.sum(),len(slwc_t_ref),100*slwc_t_ref.mean()))
    print('Ref: {:.0f} time steps have SLCC, out of {} cloudy time steps ({:.1f}%)'.format(slwc_t_ref[pks_t].sum(),len(slwc_t_ref[pks_t]),100*slwc_t_ref[pks_t].mean()))
    print('Model: {:.0f} time steps have SLCC, out of {} total time steps ({:.1f}%)'.format(slwc_t_mod.sum(),len(slwc_t_mod),100*slwc_t_mod.mean()))
    print('Model: {:.0f} time steps have SLCC, out of {} cloudy time steps ({:.1f}%)'.format(slwc_t_mod[pks_t].sum(),len(slwc_t_mod[pks_t]),100*slwc_t_mod[pks_t].mean()))

    print('Precision/Recall of SLWC classification, examining all time steps: {:.3f}/{:.3f}'.format(precision_score(slwc_t_ref, slwc_t_mod),recall_score(slwc_t_ref, slwc_t_mod)))
    print('Precision/Recall of SLWC classification, examining cloudy time steps: {:.3f}/{:.3f}'.format(precision_score(slwc_t_ref[pks_t], slwc_t_mod[pks_t]),recall_score(slwc_t_ref[pks_t], slwc_t_mod[pks_t])))


def getStats(date, model_name):

    ## Opening files
    infile_ref = reference_mask_dir.format(date[:-2])+str(date)+'.nc'
    print('  <--- ' + infile_ref)
    ds_ref = xr.open_dataset(infile_ref)

    infile_mod = model_mask_dir.format(model_name,date[:-2])+str(date)+'.nc'
    print('  <--- ' + infile_mod)
    ds_mod = xr.open_dataset(infile_mod)

    ## Selecting variables
    slwc_ref = ds_ref.slwc
    slwc_t_ref = slwc_ref.max(dim='level')

    slwc_t_mod = ds_mod.slwc_t
    class_pks = ds_mod.classified_peaks
    pks_t = ~class_pks.max(dim='level').isnull()  # True if peaks are present, false if no peaks present

    ds_ref.close()
    ds_mod.close()

    return slwc_t_ref, slwc_t_mod, pks_t

In [72]:
slwc_t_ref_all = []
slwc_t_mod_all = []
pks_t_all = []

dates = [file[-11:-3].replace('-','') for file in sorted(glob.glob(reference_mask_dir.format('*')+'/*.nc'))]

for date in dates:
    slwc_t_ref, slwc_t_mod, pks_t = getStats(date, 'model_001')

    if slwc_t_ref.sum() != 0:  # Excludes dates when temp info not available
        # printStats(slwc_t_ref, slwc_t_mod, pks_t)
        slwc_t_ref_all.append(slwc_t_ref)
        slwc_t_mod_all.append(slwc_t_mod)
        pks_t_all.append(pks_t)
    else:
        print(date, 'empty')

slwc_t_ref_cat = xr.concat(slwc_t_ref_all, dim='time')
slwc_t_mod_cat = xr.concat(slwc_t_mod_all, dim='time')
pks_t_cat = xr.concat(pks_t_all, dim='time')

print()
print(' ---> Full dataset')
printStats(slwc_t_ref_cat, slwc_t_mod_cat, pks_t_cat)


  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_reference/processed202105/20210506.nc
  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_model_processed/model_001/model_processed_202105/20210506.nc
20210506 empty
  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_reference/processed202105/20210507.nc
  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_model_processed/model_001/model_processed_202105/20210507.nc
20210507 empty
  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_reference/processed202105/20210508.nc
  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_model_processed/model_001/model_processed_202105/20210508.nc
20210508 empty
  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_reference/processed202105/20210509.nc
  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_model_processed/model_001/model_processed_202105/20210509.nc
20210509 empty
  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_reference/processed202105/20210510.nc
  <--- /hom

In [66]:
slwc_t_ref, slwc_t_mod, pks_t = getStats('20210515', 'model_001')
# printStats(slwc_t_ref, slwc_t_mod, pks_t)
if slwc_t_ref.sum() != 0:
    printStats(slwc_t_ref, slwc_t_mod, pks_t)    

  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_reference/processed202105/20210515.nc
  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_model_processed/model_001/model_processed_202105/20210515.nc
68 cloudy time steps (with peaks detected), out of 288 time steps (23.6%)
Ref: 5 time steps have SLWC, out of 288 total time steps (1.7%)
Ref: 5 time steps have SLCC, out of 68 cloudy time steps (7.4%)
Model: 4 time steps have SLCC, out of 288 total time steps (1.4%)
Model: 4 time steps have SLCC, out of 68 cloudy time steps (5.9%)
Precision/Recall of SLWC classification, examining all time steps: 0.750/0.600
Precision/Recall of SLWC classification, examining cloudy time steps: 0.750/0.600
